# Notebook 3: Train / validation / test split

In [1]:
import sys
assert "olist_mlops" in sys.executable, f"wrong kernel selected in the editor - expected the olist_mlops env, got: {sys.executable}"

In [2]:
import pandas as pd
from pathlib import Path

## read the labeled table from notebook 2

In [3]:
ARTIFACTS_DIR = Path("artifacts")

labeled_table = pd.read_csv(ARTIFACTS_DIR / "02_labeled_table.csv")

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    labeled_table[col] = pd.to_datetime(labeled_table[col])

labeled_table.shape

(96476, 26)

## split by time, not randomly

we're going to use this model on future orders, so train should be the oldest orders and test
the most recent ones. a random split would mix future orders into training and let the model
see things like holiday seasons or later carrier changes it wouldn't actually have at
prediction time. 70% oldest -> train, next 15% -> val, most recent 15% -> test.

In [4]:
labeled_table = labeled_table.sort_values("order_purchase_timestamp").reset_index(drop=True)

n = len(labeled_table)
train_cutoff = int(n * 0.70)
val_cutoff = int(n * 0.85)

train = labeled_table.iloc[:train_cutoff].copy()
val = labeled_table.iloc[train_cutoff:val_cutoff].copy()
test = labeled_table.iloc[val_cutoff:].copy()

len(train), len(val), len(test)

(67533, 14471, 14472)

## check the split - date range and label balance in each one

In [5]:
for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"--- {name} ---")
    print("rows:", len(split))
    print("date range:", split["order_purchase_timestamp"].min(), "to", split["order_purchase_timestamp"].max())
    print("late rate:", split["late"].mean().round(4))
    print()

--- train ---
rows: 67533
date range: 2016-09-15 12:16:38 to 2018-04-15 20:07:56
late rate: 0.0903

--- val ---
rows: 14471
date range: 2018-04-15 20:10:23 to 2018-06-21 07:50:39
late rate: 0.0534

--- test ---
rows: 14472
date range: 2018-06-21 08:29:29 to 2018-08-29 15:00:37
late rate: 0.0661



since this is a time based split, the late rate can drift a bit between splits (delivery
got better/worse over time, holiday season spikes, etc) - that's expected, not a bug. a random
split would've kept the same ratio everywhere but wouldn't match how the model actually gets used.

## save the artifacts

In [6]:
train.to_csv(ARTIFACTS_DIR / "03_train.csv", index=False)
val.to_csv(ARTIFACTS_DIR / "03_val.csv", index=False)
test.to_csv(ARTIFACTS_DIR / "03_test.csv", index=False)

print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)

train: (67533, 26)
val: (14471, 26)
test: (14472, 26)
